# AetherSense - Colab ML Pipeline

Notebook unico para baixar/cachear datasets publicos no Google Drive, inspecionar dados CSI, treinar o modelo offline de ocupacao/localizacao multi-pessoa e exportar ONNX para uma fase futura de integracao no backend Rust.

**Restricao etica:** este notebook nao coleta dados humanos locais. Use apenas datasets publicos/oficiais e, futuramente, calibracao de sala vazia sem pessoas.


## Ordem de execucao

1. Rode instalacao e configuracao.
2. Monte o Google Drive.
3. Configure credenciais Kaggle se quiser baixar WiMANS automaticamente.
4. Rode `FAST_DEV_MODE=True` primeiro para validar loaders, treino curto e export ONNX.
5. Rode treino completo somente depois do smoke test.

Artefatos finais esperados em `ARTIFACT_ROOT`: `occupancy.onnx`, `occupancy_metadata.json`, `metrics_summary.json`, figuras e checkpoints.


In [1]:
#@title Install dependencies
%pip -q install numpy pandas scipy scikit-learn matplotlib pillow h5py gdown kaggle zenodo-get onnx onnxscript onnxruntime torch torchvision tqdm

import importlib, sys
print('Python', sys.version)
for pkg in ['torch', 'onnx', 'onnxruntime', 'numpy', 'pandas']:
    mod = importlib.import_module(pkg)
    print(pkg, getattr(mod, '__version__', 'ok'))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 97.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 55.0 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.6/254.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.4/83.4 kB 9.1 MB/s eta 0:00:00
Python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch 2.11.0+cu128
onnx 1.21.0
onnxruntime 1.26.0
numpy 2.0.2
pandas 2.2.2


In [2]:
#@title Pipeline configuration
from pathlib import Path

RUN_ZENODO = True
RUN_WIMANS = True
RUN_WIPOSE = False
FAST_DEV_MODE = False  # first run should stay True; set False for full training

DRIVE_ROOT = Path('/content/drive/MyDrive/AetherSense_Datasets')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/AetherSense_Artifacts')

ZENODO_DIR = DRIVE_ROOT / 'zenodo_8021099'
WIMANS_DIR = DRIVE_ROOT / 'WiMANS'
WIPOSE_DIR = DRIVE_ROOT / 'Wi-Pose'

TARGET_SUBCARRIERS = 64
TARGET_TIMESTEPS = 50
HEATMAP_SIZE = (20, 20)
BATCH_SIZE = 16 if FAST_DEV_MODE else 64
ZENODO_EPOCHS = 1 if FAST_DEV_MODE else 8
WIMANS_EPOCHS = 1 if FAST_DEV_MODE else 40
LEARNING_RATE = 1e-3
RANDOM_SEED = 42

# WiMANS official annotations use discrete location labels a..e.
# These normalized anchors are used for the heatmap when metric x/y are absent.
LOCATION_ANCHORS = {
    'a': (0.18, 0.18),
    'b': (0.82, 0.18),
    'c': (0.50, 0.50),
    'd': (0.18, 0.82),
    'e': (0.82, 0.82),
}

print('FAST_DEV_MODE =', FAST_DEV_MODE)
print('DRIVE_ROOT =', DRIVE_ROOT)
print('ARTIFACT_ROOT =', ARTIFACT_ROOT)


FAST_DEV_MODE = False
DRIVE_ROOT = /content/drive/MyDrive/AetherSense_Datasets
ARTIFACT_ROOT = /content/drive/MyDrive/AetherSense_Artifacts


In [3]:
#@title Mount Google Drive and create folders
from google.colab import drive
drive.mount('/content/drive')

for path in [DRIVE_ROOT, ARTIFACT_ROOT, ZENODO_DIR, WIMANS_DIR, WIPOSE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Dataset root:', DRIVE_ROOT)
print('Artifact root:', ARTIFACT_ROOT)


Mounted at /content/drive
Dataset root: /content/drive/MyDrive/AetherSense_Datasets
Artifact root: /content/drive/MyDrive/AetherSense_Artifacts


In [4]:
#@title Imports and reproducibility
import os
import json
import math
import shutil
import zipfile
import subprocess
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import h5py
import matplotlib.pyplot as plt
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
import scipy.io as sio
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix
from torch.utils.data import DataLoader, Dataset, random_split, Subset
from tqdm.auto import tqdm

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


Device: cuda


In [5]:
#@title Progress UI helpers
from contextlib import contextmanager

TQDM_BAR_FORMAT = '{l_bar}{bar:32}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}] {postfix}'

@contextmanager
def phase_progress(title: str, total: int):
    """Pretty, consistent progress bar for each notebook phase."""
    total = max(int(total), 1)
    bar = tqdm(
        total=total,
        desc=title,
        bar_format=TQDM_BAR_FORMAT,
        colour='cyan',
        leave=True,
        dynamic_ncols=True,
    )

    def step(message: str = ''):
        if message:
            bar.set_postfix_str(message, refresh=True)
        if bar.n < bar.total:
            bar.update(1)

    failed = False
    try:
        yield step
    except Exception as exc:
        failed = True
        bar.set_postfix_str(f'failed: {type(exc).__name__}', refresh=True)
        raise
    finally:
        if not failed and bar.n < bar.total:
            bar.update(bar.total - bar.n)
        bar.close()

def progress_iter(iterable, desc: str, *, colour: str = 'cyan', leave: bool = False):
    return tqdm(
        iterable,
        desc=desc,
        bar_format=TQDM_BAR_FORMAT,
        colour=colour,
        leave=leave,
        dynamic_ncols=True,
    )


In [6]:
#@title Dataset download/cache helpers
def run_cmd(cmd: Sequence[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    print('+', ' '.join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, check=check, text=True)

def unzip_all(folder: Path) -> None:
    archives = list(folder.glob('*.zip'))
    if not archives:
        return
    with phase_progress('Extract archives', len(archives)) as step:
        for archive in archives:
            marker = folder / (archive.stem + '.unzipped')
            if marker.exists():
                step(f'{archive.name}: cached')
                continue
            print('Extracting', archive.name)
            with zipfile.ZipFile(archive) as zf:
                zf.extractall(folder)
            marker.write_text(datetime.now(timezone.utc).isoformat())
            step(f'{archive.name}: extracted')

def ensure_zenodo() -> bool:
    if any(ZENODO_DIR.rglob('*.png')) and any(ZENODO_DIR.rglob('*Labels.csv')):
        print('Zenodo already present:', ZENODO_DIR)
        return True
    print('Attempting Zenodo download with zenodo-get. If it fails, upload/extract record 8021099 manually into:', ZENODO_DIR)
    try:
        run_cmd(['zenodo_get', '10.5281/zenodo.8021099', '-o', ZENODO_DIR], check=True)
        unzip_all(ZENODO_DIR)
        return any(ZENODO_DIR.rglob('*.png')) and any(ZENODO_DIR.rglob('*Labels.csv'))
    except Exception as exc:
        print('Zenodo auto-download failed:', exc)
        print('Manual fallback: place PNG folders and train/test label CSVs under', ZENODO_DIR)
        return False

def ensure_kaggle_credentials() -> bool:
    kaggle_json = Path('/content/kaggle.json')
    drive_kaggle_json = DRIVE_ROOT / 'kaggle.json'
    target = Path.home() / '.kaggle' / 'kaggle.json'
    if target.exists():
        return True
    target.parent.mkdir(parents=True, exist_ok=True)
    if drive_kaggle_json.exists():
        shutil.copy2(drive_kaggle_json, target)
    elif kaggle_json.exists():
        shutil.copy2(kaggle_json, target)
    else:
        try:
            from google.colab import userdata
            username = userdata.get('KAGGLE_USERNAME')
            key = userdata.get('KAGGLE_KEY')
        except Exception:
            username = None
            key = None
        if username and key:
            target.write_text(json.dumps({'username': username, 'key': key}))
        else:
            print('Kaggle credentials missing; WiMANS download skipped for now.')
            print('Provide one of these, then rerun this cell:')
            print('  1) Colab Secrets: KAGGLE_USERNAME and KAGGLE_KEY')
            print('  2) Upload kaggle.json to /content/kaggle.json')
            print('  3) Save kaggle.json at', DRIVE_ROOT / 'kaggle.json')
            print('Alternative: manually extract WiMANS so annotation.csv and wifi_csi/amp/*.npy exist under', WIMANS_DIR)
            return False
    target.chmod(0o600)
    return True

def ensure_wimans() -> bool:
    ann_cached = next(WIMANS_DIR.glob('**/annotation.csv'), None)
    amp_cached = next(WIMANS_DIR.glob('**/wifi_csi/amp'), None) or next(WIMANS_DIR.glob('**/amp'), None)
    if ann_cached and amp_cached:
        print('WiMANS already present:', WIMANS_DIR)
        return True
    if not ensure_kaggle_credentials():
        return False
    try:
        run_cmd(['kaggle', 'datasets', 'download', '-d', 'shuokanghuang/wimans', '-p', WIMANS_DIR, '--unzip'], check=True)
    except Exception as exc:
        print('WiMANS Kaggle download failed:', exc)
        print('Manual fallback: extract WiMANS under', WIMANS_DIR)
        return False
    nested = next(WIMANS_DIR.glob('**/annotation.csv'), None)
    if nested and nested.parent != WIMANS_DIR:
        print('WiMANS extracted under nested folder:', nested.parent)
    print('WiMANS ready:', WIMANS_DIR)
    return next(WIMANS_DIR.glob('**/annotation.csv'), None) is not None

def ensure_wipose() -> bool:
    if any(WIPOSE_DIR.rglob('*.mat')):
        print('Wi-Pose already present:', WIPOSE_DIR)
        return True
    try:
        run_cmd(['gdown', '--fuzzy', 'https://drive.google.com/file/d/1WL6bJ-rSVdsclRt9RFc0l5hhtXYmfNg9/view?usp=sharing', '-O', WIPOSE_DIR / 'wipose_dataset.zip'], check=True)
        unzip_all(WIPOSE_DIR)
        return any(WIPOSE_DIR.rglob('*.mat'))
    except Exception as exc:
        print('Wi-Pose download failed:', exc)
        return False

dataset_tasks = []
if RUN_ZENODO:
    dataset_tasks.append(('Zenodo', ensure_zenodo))
if RUN_WIMANS:
    dataset_tasks.append(('WiMANS', ensure_wimans))
if RUN_WIPOSE:
    dataset_tasks.append(('Wi-Pose', ensure_wipose))

with phase_progress('Dataset cache/download', len(dataset_tasks) or 1) as step:
    if not dataset_tasks:
        step('all dataset phases disabled')
    for name, task in dataset_tasks:
        ready = task()
        step(f'{name}: {"ready" if ready else "skipped"}')


Dataset cache/download:   0%|                                | 0/2 [00:00<?, ?it/s] 

Zenodo already present: /content/drive/MyDrive/AetherSense_Datasets/zenodo_8021099
WiMANS already present: /content/drive/MyDrive/AetherSense_Datasets/WiMANS


In [7]:
#@title Dataset EDA / format inspection
def count_files(root: Path, suffix: str) -> int:
    return sum(1 for _ in root.rglob(f'*{suffix}'))

with phase_progress('EDA / format inspection', 3) as step:
    eda_summary = {
        'zenodo_png': count_files(ZENODO_DIR, '.png'),
        'zenodo_csv': count_files(ZENODO_DIR, '.csv'),
        'wimans_npy': count_files(WIMANS_DIR, '.npy'),
        'wimans_mat': count_files(WIMANS_DIR, '.mat'),
        'wipose_mat': count_files(WIPOSE_DIR, '.mat'),
    }
    step('file inventory')
    print(json.dumps(eda_summary, indent=2))

    ann_path = next(WIMANS_DIR.glob('**/annotation.csv'), None)
    step('annotation lookup')
    if ann_path:
        ann = pd.read_csv(ann_path, dtype=str)
        display(ann.head())
        print('WiMANS columns:', list(ann.columns))
        if 'number_of_users' in ann.columns:
            display(ann['number_of_users'].value_counts(dropna=False).sort_index())
    else:
        print('WiMANS annotation.csv not found yet; download/cache phase must complete first.')
    step('EDA complete')


EDA / format inspection:   0%|                                | 0/3 [00:00<?, ?it/s] 

{
  "zenodo_png": 1558,
  "zenodo_csv": 16,
  "wimans_npy": 11286,
  "wimans_mat": 1733,
  "wipose_mat": 0
}


,#,label,environment,wifi_band,number_of_users,user_1_location,user_2_location,user_3_location,user_4_location,user_5_location,user_6_location,user_1_activity,user_2_activity,user_3_activity,user_4_activity,user_5_activity,user_6_activity
0,1,act_1_1,classroom,2.4,1,a,NaN,NaN,NaN,NaN,NaN,nothing,NaN,NaN,NaN,NaN,NaN
1,2,act_1_2,classroom,2.4,1,a,NaN,NaN,NaN,NaN,NaN,walk,NaN,NaN,NaN,NaN,NaN
2,3,act_1_3,classroom,2.4,1,a,NaN,NaN,NaN,NaN,NaN,rotation,NaN,NaN,NaN,NaN,NaN
3,4,act_1_4,classroom,2.4,1,a,NaN,NaN,NaN,NaN,NaN,jump,NaN,NaN,NaN,NaN,NaN
4,5,act_1_5,classroom,2.4,1,a,NaN,NaN,NaN,NaN,NaN,wave,NaN,NaN,NaN,NaN,NaN


WiMANS columns: ['#', 'label', 'environment', 'wifi_band', 'number_of_users', 'user_1_location', 'user_2_location', 'user_3_location', 'user_4_location', 'user_5_location', 'user_6_location', 'user_1_activity', 'user_2_activity', 'user_3_activity', 'user_4_activity', 'user_5_activity', 'user_6_activity']


,count
number_of_users,
0,594
1,3564
2,1782
3,1782
4,1782
5,1782


In [8]:
#@title Preprocessing utilities
def pad_or_trim_time(matrix: np.ndarray, target_t: int = TARGET_TIMESTEPS) -> np.ndarray:
    if matrix.shape[1] > target_t:
        return matrix[:, -target_t:]
    if matrix.shape[1] < target_t:
        pad = np.zeros((matrix.shape[0], target_t - matrix.shape[1]), dtype=matrix.dtype)
        return np.concatenate([pad, matrix], axis=1)
    return matrix

def resample_subcarriers(matrix: np.ndarray, target_s: int = TARGET_SUBCARRIERS) -> np.ndarray:
    if matrix.shape[0] == target_s:
        return matrix
    src_x = np.linspace(0.0, 1.0, matrix.shape[0], dtype=np.float32)
    dst_x = np.linspace(0.0, 1.0, target_s, dtype=np.float32)
    out = np.empty((target_s, matrix.shape[1]), dtype=np.float32)
    for t in range(matrix.shape[1]):
        out[:, t] = np.interp(dst_x, src_x, matrix[:, t]).astype(np.float32)
    return out

def csi_array_to_subcarrier_time(arr: np.ndarray) -> np.ndarray:
    arr = np.nan_to_num(np.asarray(arr, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    if arr.ndim == 2:
        matrix = arr
        if matrix.shape[0] > matrix.shape[1]:
            matrix = matrix.T
    else:
        shape = arr.shape
        time_axis = int(np.argmax(shape))
        candidate_axes = [i for i, size in enumerate(shape) if size in (30, 52, 56, 64, 114, 128)]
        sub_axis = candidate_axes[0] if candidate_axes else int(np.argmin([abs(s - 64) for s in shape]))
        if sub_axis == time_axis and len(shape) > 1:
            sub_axis = 1 if time_axis != 1 else 0
        moved = np.moveaxis(arr, [sub_axis, time_axis], [0, 1])
        reduce_axes = tuple(range(2, moved.ndim))
        matrix = moved.mean(axis=reduce_axes) if reduce_axes else moved
    matrix = resample_subcarriers(matrix.astype(np.float32))
    matrix = pad_or_trim_time(matrix)
    baseline = matrix.mean(axis=1, keepdims=True)
    std = matrix.std(axis=1, keepdims=True) + 1e-6
    return ((matrix - baseline) / std).astype(np.float32)

def gaussian_heatmap(points: Sequence[Tuple[float, float]], size: Tuple[int, int] = HEATMAP_SIZE, sigma: float = 1.5) -> np.ndarray:
    h, w = size
    yy, xx = np.mgrid[0:h, 0:w]
    heatmap = np.zeros((h, w), dtype=np.float32)
    for x_norm, y_norm in points:
        if not (np.isfinite(x_norm) and np.isfinite(y_norm)):
            continue
        cx = np.clip(x_norm, 0.0, 1.0) * (w - 1)
        cy = np.clip(y_norm, 0.0, 1.0) * (h - 1)
        heatmap = np.maximum(heatmap, np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * sigma ** 2)))
    return heatmap.astype(np.float32)

def heatmap_to_persons(heatmap: np.ndarray, threshold: float = 0.35, max_people: int = 5) -> List[Dict[str, float]]:
    hm = heatmap.copy()
    persons = []
    for _ in range(max_people):
        y, x = np.unravel_index(np.argmax(hm), hm.shape)
        conf = float(hm[y, x])
        if conf < threshold:
            break
        persons.append({'x': float(x / max(1, hm.shape[1] - 1)), 'y': float(y / max(1, hm.shape[0] - 1)), 'confidence': conf})
        y0, y1 = max(0, y - 2), min(hm.shape[0], y + 3)
        x0, x1 = max(0, x - 2), min(hm.shape[1], x + 3)
        hm[y0:y1, x0:x1] = 0.0
    return persons


In [9]:
#@title Dataset classes
class ZenodoPresenceDataset(Dataset):
    def __init__(self, root: Path, split: str = 'train'):
        self.root = Path(root)
        label_candidates = list(self.root.rglob(f'{split}Labels.csv')) + list(self.root.rglob(f'{split}_labels.csv'))
        if not label_candidates:
            label_candidates = list(self.root.rglob('*Labels.csv'))
        if not label_candidates:
            raise FileNotFoundError(f'No Zenodo label CSV found under {self.root}')
        self.labels = pd.read_csv(label_candidates[0], header=None, names=['image_index', 'label'])
        self.images = {p.stem: p for p in self.root.rglob('*.png')}
        self.mean = 0.447
        self.std = 0.162

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        row = self.labels.iloc[idx]
        image_key = str(row['image_index'])
        image_path = self.images.get(Path(image_key).stem)
        if image_path is None:
            image_path = self.images.get(image_key)
        if image_path is None:
            raise FileNotFoundError(f'Missing Zenodo PNG for index {image_key}')
        image = Image.open(image_path).convert('L').resize((64, 64))
        arr = np.asarray(image, dtype=np.float32) / 255.0
        arr = (arr - self.mean) / self.std
        label = 0 if int(float(row['label'])) == 0 else 1
        return torch.from_numpy(arr).unsqueeze(0), torch.tensor(label, dtype=torch.long)

class WiMANSDataset(Dataset):
    def __init__(self, root: Path):
        self.root = Path(root)
        ann_path = next(self.root.glob('**/annotation.csv'), None)
        if ann_path is None:
            raise FileNotFoundError(f'No WiMANS annotation.csv found under {self.root}')
        self.annotations = pd.read_csv(ann_path, dtype=str)
        self.amp_dir = next(self.root.glob('**/wifi_csi/amp'), None) or next(self.root.glob('**/amp'), None)
        if self.amp_dir is None:
            raise FileNotFoundError(f'No WiMANS CSI amplitude folder found under {self.root}')
        self.location_cols = [f'user_{i}_location' for i in range(1, 7) if f'user_{i}_location' in self.annotations.columns]

    def __len__(self):
        return len(self.annotations)

    def _label(self, row: pd.Series) -> str:
        for col in ['label', 'sample_id', 'id']:
            if col in row and isinstance(row[col], str) and row[col].strip():
                return row[col]
        raise KeyError('WiMANS row has no label/sample_id/id column')

    def _count(self, row: pd.Series) -> int:
        if 'number_of_users' in row and pd.notna(row['number_of_users']):
            return int(float(row['number_of_users']))
        if 'user_count' in row and pd.notna(row['user_count']):
            return int(float(row['user_count']))
        return sum(1 for col in self.location_cols if str(row.get(col, 'nan')).lower() != 'nan')

    def _locations(self, row: pd.Series) -> List[Tuple[float, float]]:
        points = []
        numeric_pairs = [('x', 'y'), ('loc_x', 'loc_y'), ('location_x', 'location_y')]
        for x_col, y_col in numeric_pairs:
            if x_col in row and y_col in row and pd.notna(row[x_col]) and pd.notna(row[y_col]):
                points.append((float(row[x_col]), float(row[y_col])))
        if points:
            return points
        for col in self.location_cols:
            label = str(row.get(col, 'nan')).strip().lower()
            if label in LOCATION_ANCHORS:
                points.append(LOCATION_ANCHORS[label])
        return points

    def __getitem__(self, idx):
        row = self.annotations.iloc[idx]
        label = self._label(row)
        arr = np.load(self.amp_dir / f'{label}.npy')
        matrix = csi_array_to_subcarrier_time(arr)
        count = min(self._count(row), 3)
        points = self._locations(row)
        heatmap = gaussian_heatmap(points)
        return (
            torch.from_numpy(matrix).unsqueeze(0),
            torch.tensor(count, dtype=torch.long),
            torch.from_numpy(heatmap).unsqueeze(0),
        )

def maybe_subset(dataset: Dataset, max_items: int) -> Dataset:
    if not FAST_DEV_MODE or len(dataset) <= max_items:
        return dataset
    return Subset(dataset, list(range(max_items)))


In [10]:
#@title Model definitions
class PresenceCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Linear(64, 2)

    def forward(self, x):
        return self.fc(self.net(x).flatten(1))

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class OccupancyHeatmapNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = ConvBlock(1, 16)
        self.enc2 = ConvBlock(16, 32)
        self.enc3 = ConvBlock(32, 64)
        self.bottleneck = ConvBlock(64, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = ConvBlock(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = ConvBlock(64, 32)
        self.heat = nn.Conv2d(32, 1, 1)
        self.count_head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 4))

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1, 2))
        e3 = self.enc3(F.max_pool2d(e2, 2))
        b = self.bottleneck(F.max_pool2d(e3, 2))
        count_logits = self.count_head(b)
        d2 = self.up2(b)
        d2 = F.interpolate(d2, size=e3.shape[-2:], mode='bilinear', align_corners=False)
        d2 = self.dec2(torch.cat([d2, e3], dim=1))
        d1 = self.up1(d2)
        d1 = F.interpolate(d1, size=e2.shape[-2:], mode='bilinear', align_corners=False)
        d1 = self.dec1(torch.cat([d1, e2], dim=1))
        heatmap = torch.sigmoid(F.interpolate(self.heat(d1), size=HEATMAP_SIZE, mode='bilinear', align_corners=False))
        return count_logits, heatmap

def parameter_count(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print('Occupancy params:', parameter_count(OccupancyHeatmapNet()))


Occupancy params: 482613


In [11]:
#@title Training and evaluation utilities
def split_dataset(dataset: Dataset, train_frac: float = 0.8):
    train_len = int(len(dataset) * train_frac)
    val_len = len(dataset) - train_len
    return random_split(dataset, [train_len, val_len], generator=torch.Generator().manual_seed(RANDOM_SEED))

def train_presence(dataset: Dataset) -> Dict[str, float]:
    dataset = maybe_subset(dataset, 256)
    train_ds, val_ds = split_dataset(dataset)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    model = PresenceCNN().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    with phase_progress('Zenodo presence training', ZENODO_EPOCHS + 1) as step:
        for epoch in range(ZENODO_EPOCHS):
            model.train()
            for x, y in progress_iter(train_loader, f'Zenodo batches {epoch + 1}/{ZENODO_EPOCHS}', colour='green'):
                x, y = x.to(DEVICE), y.to(DEVICE)
                loss = F.cross_entropy(model(x), y)
                opt.zero_grad(); loss.backward(); opt.step()
            step(f'epoch {epoch + 1}/{ZENODO_EPOCHS}')

        preds, labels = [], []
        model.eval()
        with torch.no_grad():
            for x, y in progress_iter(val_loader, 'Zenodo validation', colour='blue'):
                logits = model(x.to(DEVICE))
                preds.extend(logits.argmax(1).cpu().tolist())
                labels.extend(y.tolist())
        acc = accuracy_score(labels, preds) if labels else 0.0
        step(f'val acc={acc:.3f}')

    torch.save(model.state_dict(), ARTIFACT_ROOT / 'presence_baseline.pt')
    return {'presence_val_accuracy': float(acc)}

def train_occupancy(dataset: Dataset) -> Tuple[nn.Module, Dict[str, float]]:
    dataset = maybe_subset(dataset, 512)
    train_ds, val_ds = split_dataset(dataset)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    model = OccupancyHeatmapNet().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    with phase_progress('WiMANS occupancy training', WIMANS_EPOCHS + 1) as step:
        for epoch in range(WIMANS_EPOCHS):
            model.train()
            losses = []
            for x, count, heat in progress_iter(train_loader, f'WiMANS batches {epoch + 1}/{WIMANS_EPOCHS}', colour='magenta'):
                x, count, heat = x.to(DEVICE), count.to(DEVICE), heat.to(DEVICE)
                logits, pred_heat = model(x)
                loss_count = F.cross_entropy(logits, count)
                loss_heat = F.mse_loss(pred_heat, heat)
                loss = loss_heat + 0.5 * loss_count
                opt.zero_grad(); loss.backward(); opt.step()
                losses.append(float(loss.detach().cpu()))
            step(f'epoch {epoch + 1}/{WIMANS_EPOCHS}, loss={np.mean(losses):.4f}' if losses else f'epoch {epoch + 1}/{WIMANS_EPOCHS}')

        y_true, y_pred, heat_mse = [], [], []
        model.eval()
        with torch.no_grad():
            for x, count, heat in progress_iter(val_loader, 'WiMANS validation', colour='blue'):
                logits, pred_heat = model(x.to(DEVICE))
                y_true.extend(count.tolist())
                y_pred.extend(logits.argmax(1).cpu().tolist())
                heat_mse.append(float(F.mse_loss(pred_heat.cpu(), heat).item()))
        step('validation complete')

    metrics = {
        'count_val_accuracy': float(accuracy_score(y_true, y_pred)) if y_true else 0.0,
        'heatmap_val_mse': float(np.mean(heat_mse)) if heat_mse else 0.0,
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3]).tolist() if y_true else [],
    }
    torch.save(model.state_dict(), ARTIFACT_ROOT / 'occupancy_heatmap.pt')
    return model, metrics


In [12]:
#@title Phase 1 - Zenodo ESP32 presence baseline
metrics_summary = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'fast_dev_mode': FAST_DEV_MODE,
    'target_input_shape': [1, 1, TARGET_SUBCARRIERS, TARGET_TIMESTEPS],
}

if RUN_ZENODO and any(ZENODO_DIR.rglob('*.png')):
    zenodo_ds = ZenodoPresenceDataset(ZENODO_DIR, split='train')
    print('Zenodo samples:', len(zenodo_ds))
    metrics_summary.update(train_presence(zenodo_ds))
else:
    print('Skipping Zenodo phase: dataset not available or RUN_ZENODO=False')


Zenodo samples: 308


Zenodo presence training:   0%|                                | 0/9 [00:00<?, ?it/s] 

Zenodo batches 1/8:   0%|                                | 0/4 [00:00<?, ?it/s] 

Zenodo batches 2/8:   0%|                                | 0/4 [00:00<?, ?it/s] 

Zenodo batches 3/8:   0%|                                | 0/4 [00:00<?, ?it/s] 

...

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
^    ^if w.is_alive():
^  ^^ 
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
     assert self._parent_pid == os.getpid(), 'can only test a child process' 
  ^^ ^ ^ ^ ^ ^^  ^  ^^ ^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^  ^ ^ ^ ^ ^ ^   ^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^AssertionError^: ^can only test a child process^
^^^^^^^^
AssertionError: can only test a child process


Zenodo batches 7/8:   0%|                                | 0/4 [00:00<?, ?it/s] 

Zenodo batches 8/8:   0%|                                | 0/4 [00:00<?, ?it/s] 

Zenodo validation:   0%|                                | 0/1 [00:00<?, ?it/s] 

Exception in thread QueueFeederThread:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
    reader_close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 291, in _feed
    queue_sem.release()
ValueError: semaphore or lock released too many times


In [13]:
#@title Phase 2 - WiMANS count + location heatmap
occupancy_model = None
wimans_annotation = next(WIMANS_DIR.glob('**/annotation.csv'), None)
if RUN_WIMANS and wimans_annotation is not None:
    wimans_ds = WiMANSDataset(WIMANS_DIR)
    print('WiMANS samples:', len(wimans_ds))
    sample_x, sample_count, sample_heat = wimans_ds[0]
    print('Sample input:', tuple(sample_x.shape), 'count:', int(sample_count), 'heatmap:', tuple(sample_heat.shape))
    occupancy_model, wimans_metrics = train_occupancy(wimans_ds)
    metrics_summary.update(wimans_metrics)
else:
    print('Skipping WiMANS phase: dataset not available or RUN_WIMANS=False')


WiMANS samples: 11286
Sample input: (1, 64, 50) count: 1 heatmap: (1, 20, 20)


WiMANS occupancy training:   0%|                                | 0/41 [00:00<?, ?it/s] 

WiMANS batches 1/40:   0%|                                | 0/142 [00:00<?, ?it/s] 

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79bcfea95800>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79bcfea95800>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

KeyboardInterrupt: 

In [ ]:
#@title Phase 3 - Optional Wi-Pose inspection/pretraining hook
with phase_progress('Wi-Pose optional phase', 2) as step:
    if RUN_WIPOSE:
        mat_path = next(WIPOSE_DIR.rglob('*.mat'), None)
        step('locating .mat')
        if mat_path is None:
            raise FileNotFoundError('RUN_WIPOSE=True but no .mat file was found')
        print('Inspecting Wi-Pose file:', mat_path)
        try:
            mat = sio.loadmat(mat_path)
            print('MAT keys:', [k for k in mat.keys() if not k.startswith('__')])
        except NotImplementedError:
            with h5py.File(mat_path, 'r') as f:
                print('HDF5 keys:', list(f.keys()))
        print('Wi-Pose pretraining is intentionally a documented optional hook in this v1 notebook.')
        step('inspection complete')
    else:
        step('disabled by config')
        print('Wi-Pose disabled by default to keep Colab runtime small.')
        step('skipped')


In [ ]:
#@title Export ONNX and validate PyTorch vs ONNXRuntime
with phase_progress('ONNX export/validation', 6) as step:
    if occupancy_model is None:
        print('No occupancy_model trained. Building an untrained model only for export smoke test.')
        occupancy_model = OccupancyHeatmapNet().to(DEVICE)
    step('model ready')

    occupancy_model.eval()
    onnx_path = ARTIFACT_ROOT / 'occupancy.onnx'
    metadata_path = ARTIFACT_ROOT / 'occupancy_metadata.json'
    metrics_path = ARTIFACT_ROOT / 'metrics_summary.json'
    dummy = torch.randn(1, 1, TARGET_SUBCARRIERS, TARGET_TIMESTEPS, device=DEVICE)

    with torch.no_grad():
        pt_logits, pt_heat = occupancy_model(dummy)
    step('PyTorch forward')

    torch.onnx.export(
        occupancy_model,
        dummy,
        onnx_path,
        input_names=['csi'],
        output_names=['count_logits', 'heatmap'],
        opset_version=17,
        dynamo=False,
        do_constant_folding=True,
    )
    step('ONNX exported')

    onnx.checker.check_model(onnx.load(onnx_path))
    step('ONNX checker passed')

    session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    ort_logits, ort_heat = session.run(None, {'csi': dummy.detach().cpu().numpy().astype(np.float32)})
    logit_diff = float(np.max(np.abs(pt_logits.detach().cpu().numpy() - ort_logits)))
    heat_diff = float(np.max(np.abs(pt_heat.detach().cpu().numpy() - ort_heat)))
    metrics_summary.update({'onnx_logit_max_abs_diff': logit_diff, 'onnx_heatmap_max_abs_diff': heat_diff})
    step(f'ORT diff logits={logit_diff:.2e}, heat={heat_diff:.2e}')

    metadata = {
        'model_name': 'aethersense_occupancy_heatmap',
        'created_at': metrics_summary['created_at'],
        'input_name': 'csi',
        'input_shape': [1, 1, TARGET_SUBCARRIERS, TARGET_TIMESTEPS],
        'outputs': ['count_logits', 'heatmap'],
        'count_classes': ['0', '1', '2', '3+'],
        'heatmap_size': list(HEATMAP_SIZE),
        'location_anchors': LOCATION_ANCHORS,
        'ethical_constraint': 'No local human dataset collection; public datasets plus future empty-room calibration only.',
    }
    metadata_path.write_text(json.dumps(metadata, indent=2))
    metrics_path.write_text(json.dumps(metrics_summary, indent=2))
    step('artifacts saved')

print('Saved:', onnx_path)
print('Saved:', metadata_path)
print('Saved:', metrics_path)
print(json.dumps(metrics_summary, indent=2))


In [ ]:
#@title Visualize one predicted heatmap and decoded person estimates
with phase_progress('Heatmap visualization', 3) as step:
    with torch.no_grad():
        logits, heat = occupancy_model(dummy)
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()[0]
        heat_np = heat.detach().cpu().numpy()[0, 0]
    step('inference complete')

    persons = heatmap_to_persons(heat_np)
    step(f'{len(persons)} peaks decoded')

    plt.figure(figsize=(5, 4))
    plt.imshow(heat_np, cmap='magma', origin='lower')
    plt.colorbar(label='confidence')
    plt.title('Predicted occupancy heatmap')
    plt.show()
    step('plot rendered')

print('Count probabilities [0, 1, 2, 3+]:', probs.round(4).tolist())
print('Decoded persons:', persons)


## Acceptance checklist

- Dataset cache exists in Google Drive.
- Smoke run passes with `FAST_DEV_MODE=True`.
- Full run is executed with `FAST_DEV_MODE=False` after smoke validation.
- ONNX export uses fixed input shape `[1, 1, 64, 50]`.
- ONNXRuntime differences are recorded in `metrics_summary.json`.
- `occupancy.onnx`, `occupancy_metadata.json`, and `metrics_summary.json` are saved in Drive.
- Backend/frontend implementation starts only after these artifacts are reviewed.
